# Kaggle Model Lab — Select · Benchmark · Serve

**Purpose:** select one or more attached model artifacts and a CSV dataset, benchmark on a shared labeled dataset, and simulate **FastAPI multi-model serving** from the same Kaggle notebook.

**Supported artifacts:** scikit-learn / joblib / pickle pipelines (`.joblib`, `.pkl`, `.pickle`), native XGBoost Booster (`.json`, `.ubj`), TorchScript (`.pt`, `.jit`, `.ts`). Ordinary PyTorch `state_dict` checkpoints require a matching `torch.nn.Module` definition; see the custom loader cell below. **Never load untrusted pickle files.**

**Important:** A model's *training dataset* is not necessarily its *evaluation dataset*. For head-to-head metrics, all selected models must support the same feature schema, task, preprocessing, and target; never benchmark on rows used for training.

**Run steps:** (1) Kaggle **Add Input → Models / Datasets**, (2) run all cells, (3) pick artifacts and CSV in the interactive menus, (4) set task/target, (5) **Load selected models**, (6) **Benchmark**, (7) run API simulation. Files are saved in `/kaggle/working` in Kaggle or in `./outputs` locally.


In [ ]:
# Dependencies (typically already installed on Kaggle).
# Run if needed, then restart the kernel if packages were newly installed:
# %pip install -q fastapi uvicorn httpx ipywidgets xgboost scikit-learn joblib
# If the import of ipywidgets below fails, uncomment the line above and run it.
# Package installation may require an Internet-enabled Kaggle session.

import sys, importlib.util
for pkg in ('pandas','numpy','fastapi','httpx','ipywidgets','sklearn','xgboost','torch'):
    print(f'{pkg:12}', 'available' if importlib.util.find_spec(pkg) else 'NOT INSTALLED')


In [ ]:
import os, json, time, threading, warnings, traceback
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets

ROOTS = [Path('/kaggle/input'), Path('./models'), Path('./datasets')]
OUTPUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
OUTPUT.mkdir(exist_ok=True, parents=True)
MODEL_EXTS={'.joblib','.pkl','.pickle','.ubj','.jit','.ts','.pt'}
DATA_EXTS={'.csv'}

def scan(exts):
    results=[]
    for root in ROOTS:
        if root.exists():
            results.extend(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in exts)
    return sorted(set(results), key=lambda x: str(x))

# XGBoost JSON files often contain either a model or unrelated metadata.
# Include all JSON candidates, but use the loader to reject invalid models.
def model_scan():
    return sorted(set(scan(MODEL_EXTS) + [p for p in scan({'.json'}) if p.name.lower() not in {'metadata.json','dataset-metadata.json','kernel-metadata.json'}]), key=str)

model_files=model_scan()
csv_files=scan(DATA_EXTS)
print('Detected model artifacts:', len(model_files))
print('Detected CSV datasets: ', len(csv_files))
for p in model_files[:20]: print(' MODEL', p)
for p in csv_files[:15]: print(' DATA ', p)
if not model_files: print('Attach Kaggle Models via Add Input, or place local files in ./models')
if not csv_files: print('Attach a CSV Dataset via Add Input, or place local files in ./datasets')


## Choose models and dataset

Select **multiple models** to benchmark; choose **one model** to inspect or query. `XGBoost` native Booster files require the exact training feature order. A fitted scikit-learn preprocessing pipeline serialized together with the estimator is usually easier to reuse correctly.

In [ ]:
model_picker=widgets.SelectMultiple(
    options=[(str(p),str(p)) for p in model_files],
    value=tuple(str(p) for p in model_files[:2]),
    description='Models', rows=min(max(len(model_files),3),10), layout=widgets.Layout(width='95%'))
csv_picker=widgets.Dropdown(options=[(str(p),str(p)) for p in csv_files], description='Dataset',layout=widgets.Layout(width='95%'))
task_picker=widgets.Dropdown(options=['classification','regression'], description='Task')
target_picker=widgets.Dropdown(options=[],description='Target')
refresh_btn=widgets.Button(description='Refresh files',button_style='info')
preview_btn=widgets.Button(description='Preview CSV')
selection_out=widgets.Output()

def refresh_files(_=None):
    global model_files,csv_files
    model_files=model_scan(); csv_files=scan(DATA_EXTS)
    model_picker.options=[(str(p),str(p)) for p in model_files]
    csv_picker.options=[(str(p),str(p)) for p in csv_files]
    if csv_files and not csv_picker.value: csv_picker.value=str(csv_files[0])

def preview_csv(_=None):
    with selection_out:
        clear_output(wait=True)
        if not csv_picker.value:
            print('No CSV attached.');return
        df0=pd.read_csv(csv_picker.value)
        target_picker.options=list(df0.columns)
        if len(df0.columns): target_picker.value=df0.columns[-1]
        print(f'Dataset shape: {df0.shape} | Columns: {list(df0.columns)}')
        display(df0.head(5))

refresh_btn.on_click(refresh_files)
preview_btn.on_click(preview_csv)
csv_picker.observe(lambda change:preview_csv() if change['name']=='value' else None, names='value')
display(model_picker,csv_picker,widgets.HBox([task_picker,target_picker]),widgets.HBox([refresh_btn,preview_btn]),selection_out)
if csv_picker.value: preview_csv()


## Load selected models

Safeguards: joblib/pickle is **unsafe for files you do not trust**; only load your own artifacts. For PyTorch, `.pt` is interpreted as **TorchScript** first. A normal `torch.save(model.state_dict(), ...)` file needs the original model architecture and cannot be served automatically. To add your architecture, replace `custom_torch_loader` below with code that constructs the model and uses `load_state_dict`.


In [ ]:
from dataclasses import dataclass
import joblib
import xgboost as xgb
import torch

@dataclass
class ModelAdapter:
    name: str
    path: str
    kind: str
    obj: Any
    expected_features: list | None = None

    def predict(self, X: pd.DataFrame):
        if self.expected_features:
            missing=[f for f in self.expected_features if f not in X.columns]
            if missing: raise ValueError(f'Missing features for {self.name}: {missing}')
            X=X.loc[:,self.expected_features]
        if self.kind=='sklearn':
            return np.asarray(self.obj.predict(X)).reshape(-1)
        if self.kind=='xgboost':
            arr=self.obj.predict(xgb.DMatrix(X, feature_names=list(X.columns)))
            arr=np.asarray(arr)
            if task_picker.value=='classification':
                return (arr>=0.5).astype(int) if arr.ndim==1 else np.argmax(arr,axis=1)
            return arr.reshape(-1)
        if self.kind=='torchscript':
            if not all(pd.api.types.is_numeric_dtype(t) for t in X.dtypes):
                raise ValueError('TorchScript demo adapter accepts numeric features only. Add your own preprocessing.')
            with torch.no_grad():
                t=torch.tensor(X.to_numpy(dtype=np.float32))
                arr=self.obj(t)
                if isinstance(arr,(tuple,list)): arr=arr[0]
                arr=arr.detach().cpu().numpy()
            if task_picker.value=='classification':
                if arr.ndim==2 and arr.shape[1]>1: return np.argmax(arr,axis=1)
                return (arr.reshape(-1)>=0.5).astype(int) # assumes logits or probabilities
            return arr.reshape(-1)
        raise ValueError('Unknown adapter kind')

def custom_torch_loader(path):
    # EXAMPLE for a state_dict checkpoint — supply YOUR actual model architecture:
    # model = MyNeuralNet(input_dim=7, hidden_dim=32)
    # state = torch.load(path, map_location='cpu', weights_only=True)
    # model.load_state_dict(state)
    # model.eval()
    # return model
    raise ValueError('State_dict detected/unsupported .pt artifact: define your architecture and loader. TorchScript .pt files are supported.')

# Extend this if you implement custom_torch_loader: add a ModelAdapter.predict branch
# or export your model to TorchScript first.
def load_artifact(path):
    p=Path(path); ext=p.suffix.lower()
    if ext in {'.joblib','.pkl','.pickle'}:
        obj=joblib.load(p)
        expected=list(obj.feature_names_in_) if hasattr(obj,'feature_names_in_') else None
        return ModelAdapter(p.stem,str(p),'sklearn',obj,expected)
    if ext in {'.json','.ubj'}:
        obj=xgb.Booster();obj.load_model(str(p))
        return ModelAdapter(p.stem,str(p),'xgboost',obj,obj.feature_names)
    if ext in {'.pt','.jit','.ts'}:
        try: obj=torch.jit.load(str(p),map_location='cpu').eval()
        except Exception as ex:
            raise ValueError(f'{p.name} is not a TorchScript file. Export as TorchScript or provide its network architecture. Original error: {ex}') from ex
        return ModelAdapter(p.stem,str(p),'torchscript',obj,None)
    raise ValueError(f'Unsupported artifact: {p}')

REGISTRY={}
load_btn=widgets.Button(description='Load selected models',button_style='success')
load_out=widgets.Output()
def load_selected(_=None):
    REGISTRY.clear()
    with load_out:
        clear_output(wait=True)
        for path in model_picker.value:
            try:
                adapter=load_artifact(path)
                # Unique even when two attached paths have the same filename
                name=adapter.name; suffix=2
                while name in REGISTRY:
                    name=f'{adapter.name}_{suffix}';suffix+=1
                adapter.name=name
                REGISTRY[name]=adapter
                print('LOADED',name,adapter.kind,'features=',adapter.expected_features)
            except Exception as exc:
                print('LOAD FAILED',Path(path).name,':',type(exc).__name__,str(exc))
        print('Loaded model count:',len(REGISTRY))
load_btn.on_click(load_selected)
display(load_btn,load_out)


## Benchmark on the chosen CSV

This is an evaluation pipeline, **not training**. Labels must already be in the CSV. Prediction columns are all columns except your selected target. For rigorous benchmarking, add a data contract to verify exact training-time column order, encoders, class names, and transformations for every model.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, mean_absolute_error, r2_score

BENCHMARK_RESULTS=None

def chosen_data():
    if not csv_picker.value: raise ValueError('Choose a CSV dataset first')
    df=pd.read_csv(csv_picker.value)
    target=target_picker.value
    if not target or target not in df: raise ValueError('Choose a valid target/label column')
    df=df.dropna(subset=[target]).copy()
    if len(df)==0: raise ValueError('No labeled rows found')
    return df.drop(columns=[target]), df[target]

def run_benchmark():
    if not REGISTRY: raise ValueError('Load at least one model first')
    X,y=chosen_data()
    rows=[]
    for name,model in REGISTRY.items():
        try:
            # one warm-up + five timed runs; total dataset latency, not per-row latency
            model.predict(X.iloc[:min(len(X),2)])
            durations=[];pred=None
            for i in range(5):
                start=time.perf_counter();pred=model.predict(X);durations.append((time.perf_counter()-start)*1000)
            if len(pred)!=len(y): raise ValueError(f'Prediction length {len(pred)} differs from dataset size {len(y)}')
            result={'model':name,'framework':model.kind,'n_rows':len(y),
                    'median_batch_latency_ms':float(np.median(durations)),
                    'avg_ms_per_row':float(np.median(durations)/len(y)),
                    'artifact_kb':round(Path(model.path).stat().st_size/1024,2), 'status':'ok'}
            if task_picker.value=='classification':
                result.update(accuracy=float(accuracy_score(y,pred)),
                              precision=float(precision_score(y,pred,average='weighted',zero_division=0)),
                              recall=float(recall_score(y,pred,average='weighted',zero_division=0)),
                              f1=float(f1_score(y,pred,average='weighted',zero_division=0)))
            else:
                result.update(mae=float(mean_absolute_error(y,pred)),
                              rmse=float(np.sqrt(mean_squared_error(y,pred))),
                              r2=float(r2_score(y,pred)))
            rows.append(result)
        except Exception as ex:
            rows.append({'model':name,'framework':model.kind,'status':'FAILED: '+str(ex)})
    result_df=pd.DataFrame(rows)
    metric='f1' if task_picker.value=='classification' else 'rmse'
    if metric in result_df: result_df=result_df.sort_values(metric,ascending=(metric=='rmse'),na_position='last')
    return result_df

benchmark_btn=widgets.Button(description='Run benchmark',button_style='primary')
benchmark_out=widgets.Output()
def benchmark_click(_):
    global BENCHMARK_RESULTS
    with benchmark_out:
        clear_output(wait=True)
        try:
            BENCHMARK_RESULTS=run_benchmark()
            display(BENCHMARK_RESULTS)
            saved=OUTPUT/'benchmark_leaderboard.csv'
            BENCHMARK_RESULTS.to_csv(saved,index=False)
            print('Saved:',saved)
        except Exception as exc:print('Benchmark failed:',str(exc))
benchmark_btn.on_click(benchmark_click)
display(benchmark_btn,benchmark_out)


## FastAPI multi-model serving (inside the notebook)

Endpoints:
- `GET /health` — status
- `GET /models` — available models
- `POST /predict/{model_name}` — `{ "instances": [{"feature_1": 1.0, ...}] }`
- `POST /predict-all` — same batch to all loaded models; reports incompatibility explicitly
- `POST /benchmark` — benchmark selected CSV (only if target labels exist)

The `TestClient` below calls **the actual FastAPI routes inside the Kaggle kernel**; no Internet, token, or public port is required. The optional localhost server is available only inside that runtime, not from your iPhone or laptop browser.


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

app=FastAPI(title='Kaggle Multi-Model Serving Lab',version='1.0.0')
class PredictRequest(BaseModel):
    instances:list[dict[str,Any]]=Field(min_length=1,max_length=10000)

@app.get('/health')
def health():return {'status':'ok','loaded_models':len(REGISTRY)}

@app.get('/models')
def models():
    return [{'name':name,'framework':m.kind,'expected_features':m.expected_features} for name,m in REGISTRY.items()]

def do_predict(m,instances):
    X=pd.DataFrame(instances)
    predictions=m.predict(X)
    return {'model':m.name,'count':len(predictions),'predictions':np.asarray(predictions).tolist()}

@app.post('/predict/{model_name}')
def predict(model_name:str,request:PredictRequest):
    if model_name not in REGISTRY:raise HTTPException(status_code=404,detail=f'Unknown model: {model_name}')
    try:return do_predict(REGISTRY[model_name],request.instances)
    except Exception as ex:raise HTTPException(status_code=422,detail=str(ex))

@app.post('/predict-all')
def predict_all(request:PredictRequest):
    return {'results':{name:(safe_predict(m,request.instances)) for name,m in REGISTRY.items()}}

def safe_predict(m,records):
    try:return do_predict(m,records)
    except Exception as ex:return {'error':str(ex)}

@app.post('/benchmark')
def api_benchmark():
    try:return {'leaderboard':run_benchmark().replace({np.nan:None}).to_dict(orient='records')}
    except Exception as ex:raise HTTPException(status_code=422,detail=str(ex))

client=TestClient(app)
print('Routes:',[r.path for r in app.routes if not r.path.startswith('/openapi') and not r.path.startswith('/docs') and not r.path.startswith('/redoc')])
print('GET /health:',client.get('/health').json())
print('GET /models:',client.get('/models').json())


## Query the API directly from Kaggle (no server required)

Click the sample button. It takes the **first selected dataset row**, strips the target column, posts it to the FastAPI API, and shows the response. This simulates actual serving without network ingress. For a manual payload, edit the example call beneath the widget.

In [ ]:
query_btn=widgets.Button(description='Query all models using first CSV row',button_style='success')
query_out=widgets.Output()
def query_click(_):
    with query_out:
        clear_output(wait=True)
        try:
            X,_=chosen_data()
            records=json.loads(X.iloc[:1].to_json(orient='records'))
            response=client.post('/predict-all',json={'instances':records})
            print('POST /predict-all',response.status_code)
            print(json.dumps(response.json(),indent=2,default=str))
        except Exception as e:print('ERROR:',e)
query_btn.on_click(query_click)
display(query_btn,query_out)

# Manual example after models are loaded:
# payload={'instances':[{'usage_score':0.3,'support_calls':2,'monthly_spend':100.0}]}
# response=client.post('/predict/YOUR_MODEL_NAME',json=payload)
# print(response.status_code, response.json())


## Optional: run a real localhost Uvicorn server

The FastAPI app above is already queryable with `TestClient`. This cell additionally starts a localhost HTTP listener in a background thread, useful for `requests` or `curl` executed **inside the same Kaggle runtime**. It is **not** a persistent public Kaggle API. Do not expose unrestricted model endpoints or inference data publicly.

In [ ]:
# Optional — uncomment and execute to enable a local HTTP socket.
# import uvicorn
# server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=8765, log_level='warning'))
# server_thread = threading.Thread(target=server.run, daemon=True)
# server_thread.start()
# import requests
# print(requests.get('http://127.0.0.1:8765/health',timeout=5).json())
# # Stop server later: server.should_exit = True


## Next stage: reproducible pipeline and deployable API

Outputs: `/kaggle/working/benchmark_leaderboard.csv` plus this notebook's output. This version deliberately does **not** republish or overwrite any Kaggle model; it only reads attached artifacts and writes benchmark results. To serve outside Kaggle, move your compatible model artifacts and FastAPI code into a Docker container and deploy using Compose/Kubernetes/Cloud Run/Azure Container Apps. If comparing fundamentally different datasets/tasks, use per-model datasets and avoid ranking incompatible metrics together.

**vLLM vs FastAPI:** vLLM is for efficient serving of compatible generative language models, not general XGBoost or PyTorch tabular classifiers. You can integrate vLLM as a third downstream backend later, with FastAPI as the routing gateway.
